# Paper figure — Coverage on Seen (Scenario 7) vs. Unseen (Scenario 9)

Companion code for:

> **Look Once, Beam Twice: Camera-Primed Real-Time Double-Directional mmWave
> Beam Management for Vehicular Connectivity**
> Avhishek Biswas\*, Apala Pramanik\*, Eylem Ekici, Mehmet C. Vuran (\*equal contribution)
> *Proc. IEEE SECON 2026*, Pisa, Italy.
> Paper (arXiv): <https://arxiv.org/pdf/2605.05071>

Reproduces **Fig. 11** of the paper: grouped coverage bars
(`coverage = 100 − outage`) per received-power quantile threshold
(Q0.80 / Q0.90 / Q0.95) for the MNet-LeNet baseline (Top-1/2/3) vs.
VIBE-YOLOR and VIBE-MA, on the **seen** scenario (7, the MNet-LeNet
training scenario) and the **unseen** scenario (9).

Inputs are the roll-up CSVs written by the two MNet-LeNet evaluation
notebooks (`MNet_LeNet_Model_Analysis/combined_outage_timing_summary_*.csv`).
Outputs PNG + PDF + SVG to `FinalPlots/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import os

# ============================== #
#   PATHS  (edit here if needed) #
# ============================== #
try:
    PROJECT_ROOT = os.path.dirname(os.path.abspath(__file__))   # run as .py
except NameError:
    PROJECT_ROOT = os.getcwd()                                  # run in Jupyter
# Roll-up CSVs come from the two MNet-LeNet evaluation notebooks:
INPUT_DIR  = os.path.join(PROJECT_ROOT, "MNet_LeNet_Model_Analysis")
# Camera-ready figures are written here:
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "FinalPlots")

# ============================== #
#   APPLY GLOBAL MATPLOTLIB RC   #
# ============================== #
plt.rcParams.update({
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'text.usetex': False,
    'font.size': 20,
    'mathtext.fontset': 'dejavusans',
    'font.family': 'DejaVu Sans'
})

# ============================== #
#       GLOBAL CONFIGURATION     #
# ============================== #
method_labels = ["LeNet_Top1", "YOLOR_Top1", "LeNet_Top2", "LeNet_Top3", "YOLOR_Corr"]
quantiles = ["0.80", "0.90", "0.95"]
quantile_labels = [f"$Q_{{{q}}}$" for q in quantiles]

colors = ["#99D6FF", "#FA7A75", "#33ADFF", "#0068AE", "#D90F08"]
hatches = ['/', '-', '//', '///', '--']

bar_width = 0.16
method_spacing = 0.18
group_spacing = 0.4
num_methods = len(method_labels)

# ============================== #
#       HELPER FUNCTIONS         #
# ============================== #
def extract_success(df):
    """Extract success rate (100 - outage) for defined methods."""
    quantile_map = {"0.80": 80, "0.90": 90, "0.95": 95}
    return np.array([
        [100 - df.loc[quantile_map[q], label] for label in method_labels]
        for q in quantiles
    ])

# ============================== #
#        LOAD DATA FILES         #
# ============================== #
df_s9 = pd.read_csv(os.path.join(INPUT_DIR, "combined_outage_timing_summary_Scenario9.csv"), index_col=0)
df_s7 = pd.read_csv(os.path.join(INPUT_DIR, "combined_outage_timing_summary_Scenario7.csv"), index_col=0)
df_s9.index = df_s9.index.astype(int)
df_s7.index = df_s7.index.astype(int)

success_s9 = extract_success(df_s9)
success_s7 = extract_success(df_s7)

scenarios = {"Seen": success_s7, "Unseen": success_s9}

# ============================== #
#         PLOTTING START         #
# ============================== #
fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.5), sharey=False)

for col, (scenario_name, success_data) in enumerate(scenarios.items()):
    all_positions, group_centers, x_offset = [], [], 0
    for _ in quantiles:
        positions = [x_offset + i * method_spacing for i in range(num_methods)]
        all_positions.append(positions)
        group_centers.append(np.mean(positions))
        x_offset = max(positions) + group_spacing
    all_positions = np.array(all_positions)

    for q_idx, positions in enumerate(all_positions):
        for m_idx, pos in enumerate(positions):
            axes[col].bar(pos, success_data[q_idx, m_idx],
                          width=bar_width, color=colors[m_idx],
                          edgecolor='white', hatch=hatches[m_idx], linewidth=0.2)

    # === Formatting per subplot ===
    ax = axes[col]
    ax.set_xticks(group_centers)
    ax.set_xticklabels(quantile_labels, fontsize=20)
    ax.set_title(scenario_name, fontsize=20)
    ax.set_ylim(0, 110)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

    if col == 0:
        ax.set_ylabel("Coverage (%)")
    else:
        ax.set_yticks([])
        ax.set_yticklabels([])
        ax.set_ylabel("")

# === Shared xlabel ===
fig.supxlabel("Quantile Based Threshold", fontsize=20,x=0.55 , y=0.2)

# ============================== #
#             LEGEND             #
# ============================== #
# Reorder so first 3 are MNet-LeNet, next 2 are VIBE methods
# Reorder so the first 3 are MNet-LeNet, then the 2 VIBE methods
custom_legend_labels = [
    "MNet-LeNet (Top-1)",
    "MNet-LeNet (Top-2)",
    "MNet-LeNet (Top-3)",
    "VIBE-YOLOR",
    "VIBE-MA"
]
# indices matching this order (LeNet Top1, Top2, Top3, then YOLOR, MA)
custom_legend_indices = [0, 2, 3, 1, 4]

legend_handles = [
    Patch(facecolor=colors[i], edgecolor='white', hatch=hatches[i], label=custom_legend_labels[j])
    for j, i in enumerate(custom_legend_indices)
]

# === Legend with 2 rows: 3 items on first row, 2 on second row ===
fig.legend(
    handles=legend_handles,
    fontsize=16,
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.55, 0.18),  # center horizontally
    ncol=3,                      # ensures row break after 3 items
    columnspacing=0.5,
    handletextpad=0.5
)

# ============================== #
#         SAVE & SHOW FIGURE     #
# ============================== #
plt.tight_layout(rect=[0, 0.08, 1, 1])
os.makedirs(OUTPUT_DIR, exist_ok=True)
_stem = os.path.join(OUTPUT_DIR, "Success_S7_S9")
plt.savefig(_stem + ".png", format="png", dpi=600, bbox_inches="tight")
plt.savefig(_stem + ".pdf", format="pdf", dpi=600, bbox_inches="tight")
plt.savefig(_stem + ".svg", format="svg", bbox_inches="tight")   # native vector for the paper
print(f"[INFO] wrote {_stem}.png / .pdf / .svg")
plt.show()
